# P16 — CCT20 dataset + baseline A (Faster R-CNN R50-FPN)

Stage 2 deliverable: working dataset code and first results of a simple baseline.

**Kaggle settings (right panel):** Accelerator → **GPU T4 x2** (one GPU is used), Internet → **On**
(needs a phone-verified Kaggle account). *Run All* takes about 2–3 hours with the default settings.

What the notebook does:
1. clones the repo and downloads CCT20 (~6 GB) straight into `/tmp` (streamed, the archive is not stored);
2. builds the leakage-free splits and condition tags;
3. runs a 2-minute smoke test of the whole pipeline;
4. evaluates the untrained COCO reference (class-agnostic);
5. fine-tunes **baseline A** and evaluates it on val → threshold → test;
6. saves everything to `/kaggle/working/stage2_outputs.zip` (splits, metrics, logs, figures) to commit to the repo.

In [ ]:
# ---- settings ----
REPO = "https://github.com/Seren1971/CV-Project.git"   # private repo: use https://<token>@github.com/...
EPOCHS = 6             # plan: 20; 6 is enough for first results (~15 min/epoch on a T4)
TRAIN_FRACTION = 1.0   # lower (e.g. 0.5) if short on GPU time
BATCH_SIZE = 4
LR = 0.01
SEED = 0
RUN_REFERENCE = True   # untrained COCO model, class-agnostic
DATA = "/tmp/cct20"

In [ ]:
import os, subprocess, sys, glob, json
os.chdir("/kaggle/working")
if not os.path.exists("CV-Project"):
    subprocess.run(["git", "clone", "--depth", "1", REPO, "CV-Project"], check=True)
os.chdir("/kaggle/working/CV-Project")
subprocess.run("git log -1 --format='%h %s'", shell=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"])
import torch, torchvision
print("torch", torch.__version__, "torchvision", torchvision.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - enable GPU!")

## 1. Download CCT20 (LILA BC)

In [ ]:
%%bash -s "$DATA"
set -e
DATA=$1; mkdir -p $DATA; cd $DATA
BASE=https://storage.googleapis.com/public-datasets-lila/caltechcameratraps
[ -n "$(ls -d */ 2>/dev/null | grep -i annot)" ] || curl -sL $BASE/eccv_18_annotations.tar.gz | tar xz
[ -n "$(ls -d */ 2>/dev/null | grep -i images)" ] || curl -sL $BASE/eccv_18_all_images_sm.tar.gz | tar xz
ls; du -sh $DATA

In [ ]:
ANN_DIR = os.path.dirname(glob.glob(f"{DATA}/**/train_annotations.json", recursive=True)[0])
IMG_DIR = max((d for d in glob.glob(f"{DATA}/*/") if "annot" not in d.lower()),
              key=lambda d: len(os.listdir(d))).rstrip("/")
print("annotations:", ANN_DIR, "\nimages:", IMG_DIR, len(os.listdir(IMG_DIR)), "files")

## 2. Splits and condition tags

In [ ]:
!python -m src.data.prepare_cct20 --ann-dir "$ANN_DIR" --out-dir data/splits --with-cis-test
!python -m src.data.tag_conditions --split test --image-dir "$IMG_DIR"
import pandas as pd
pd.read_csv("data/splits/split_stats.csv")

## 3. Smoke test (≈2 min): tiny training + inference, checks the whole pipeline before the long run

In [ ]:
!python -m src.train --epochs 1 --train-fraction 0.01 --val-max 40 --workers 2 --image-dir "$IMG_DIR" --out runs/smoke
!python -m src.predict --ckpt runs/smoke/best.pt --split test --max-images 40 --image-dir "$IMG_DIR" --out runs/smoke/preds_test.json

## 4. Reference: COCO-pretrained Faster R-CNN, no CCT20 training (class-agnostic)

In [ ]:
if RUN_REFERENCE:
    for split in ["val", "test"]:
        !python -m src.predict --coco-reference --split {split} --image-dir "$IMG_DIR" --out runs/reference/preds_{split}.json
    !python -m src.evaluate --name "COCO reference" --agnostic-only --val-preds runs/reference/preds_val.json --test-preds runs/reference/preds_test.json --out results/reference

## 5. Baseline A: fine-tuning with standard augmentation

In [ ]:
!python -m src.train --variant A --epochs {EPOCHS} --train-fraction {TRAIN_FRACTION} --batch-size {BATCH_SIZE} --lr {LR} --seed {SEED} --image-dir "$IMG_DIR" --out runs/A_s{SEED}

In [ ]:
RUN = f"runs/A_s{SEED}"
for split in ["val", "test"]:
    !python -m src.predict --ckpt {RUN}/best.pt --split {split} --image-dir "$IMG_DIR" --out {RUN}/preds_{split}.json
!python -m src.evaluate --name "Baseline A" --val-preds {RUN}/preds_val.json --test-preds {RUN}/preds_test.json --out results/baseline_A

## 6. Results

In [ ]:
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
for d in ["results/reference", "results/baseline_A"]:
    if os.path.exists(f"{d}/summary.md"):
        display(Markdown(open(f"{d}/summary.md").read()))

log = pd.DataFrame(json.load(open(f"{RUN}/log.json")))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(log.epoch, log.train_loss, marker="o"); ax[0].set(title="Train loss", xlabel="epoch")
ax[1].plot(log.epoch, log.val_map50, marker="o"); ax[1].set(title="Val mAP@0.5 (selection subset)", xlabel="epoch")
plt.tight_layout(); os.makedirs("results/baseline_A", exist_ok=True)
plt.savefig("results/baseline_A/training_curve.png", dpi=120); plt.show()
log

### Examples on test: green = ground truth, red = baseline A above the val threshold

In [ ]:
from PIL import Image, ImageDraw
from src.data.dataset import scaled_boxes
recs = {r["image_id"]: r for r in json.load(open("data/splits/test.json"))}
preds = json.load(open(f"{RUN}/preds_test.json"))
thr = json.load(open("results/baseline_A/metrics.json"))["class_aware"]["threshold_from_val"]
classes = json.load(open("data/splits/classes.json"))
tags = pd.read_csv("data/splits/test_conditions.csv").set_index("image_id")

def worst(pred, rec):  # false alarms on empty frames, then missed animals
    s = [x for x in pred["scores"] if x >= thr]
    return (max(s) if s else 0) if not rec["boxes"] else (1 - max(s) if s else 1)
ids = sorted(preds, key=lambda i: -worst(preds[i], recs[i]))[:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, i in zip(axes.ravel(), ids):
    im = Image.open(f"{IMG_DIR}/{recs[i]['file_name']}").convert("RGB"); d = ImageDraw.Draw(im)
    for x, y, w, h in scaled_boxes(recs[i], *im.size): d.rectangle([x, y, x + w, y + h], outline="lime", width=4)
    p = preds[i]
    for b, s, l in zip(p["boxes"], p["scores"], p["labels"]):
        if s >= thr: d.rectangle(b, outline="red", width=4); d.text((b[0] + 4, b[1] + 4), f"{classes[l-1]} {s:.2f}", fill="red")
    t = tags.loc[i] if i in tags.index else None
    cond = ", ".join(c for c in ["is_night", "truncated", "is_small"] if t is not None and t[c])
    ax.imshow(im); ax.axis("off"); ax.set_title(f"{recs[i]['label']} | {cond or 'no tags'}", fontsize=9)
plt.tight_layout(); plt.savefig("results/baseline_A/examples.png", dpi=100); plt.show()

## 7. Save outputs (download `stage2_outputs.zip` from the Output tab and commit its contents)

In [ ]:
!cp {RUN}/log.json {RUN}/config.json results/baseline_A/
!pip freeze | grep -i -E "^(torch|torchvision|pycocotools|numpy|pillow)==" > results/environment.txt
!nvidia-smi --query-gpu=name --format=csv,noheader >> results/environment.txt
!zip -qr /kaggle/working/stage2_outputs.zip data/splits results
!ls -la /kaggle/working/stage2_outputs.zip